# Reason or Recall? — Kaggle runner

Thin entry point: the code lives on GitHub ([umardrazbhatti-work/ReasonOrRecall](https://github.com/umardrazbhatti-work/ReasonOrRecall)); this notebook clones it and calls it. Every experiment goes through the registry-backed runner, so results land in `runs/results.jsonl` and finished experiments are never repeated.

**One-time setup (notebook editor, right-hand sidebar):**
1. **Settings → Accelerator:** `GPU T4 x2` (not P100: it lacks the 4-bit NF4 kernels QLoRA needs). **Settings → Internet:** on.
2. **Input → Add Input → Datasets → Your Datasets:** attach `ror-data` (the uploaded `ror-data.zip`).
3. **Add-ons → Secrets:** add a secret named `HF_TOKEN` (your Hugging Face token) and tick it for this notebook. Needed for gated models (Llama-3.1). Never paste the token into a cell.
4. **Continuing earlier work:** Input → Add Input → *Your Work* → this notebook → attach its latest version's output. Its `runs/` is restored below, so completed experiments are skipped and interrupted training resumes from its checkpoint.

**Run:** set `CONFIG` below. Keep `DRY_RUN = True` to only see the plan. For real runs set `DRY_RUN = False` and use **Save Version → Save & Run All (Commit)** so the run keeps going after you close the browser and `/kaggle/working` is saved as output.

In [ ]:
# ---- CONFIG -----------------------------------------------------------------
REPO_URL  = "https://github.com/umardrazbhatti-work/ReasonOrRecall.git"
GIT_REF   = "main"          # branch, tag or commit SHA (pin a SHA to reproduce a run exactly)
SUITE     = "configs/suite_phase1.yaml"
ONLY_ARM  = "A5"            # e.g. "A5"; None = every arm in the suite
MODEL     = "qwen2.5-3b"    # e.g. "qwen2.5-3b"; None = every model in the suite
SPLIT     = "standard"      # "standard" | "clean" | None (both)
SEED      = 0               # e.g. 0; None = every seed
DRY_RUN   = True            # True: plan only. False: execute the experiments.
RUN_TESTS = True            # run pytest before anything else

REPO_DIR  = "/tmp/ReasonOrRecall"      # code checkout (not saved as output)
RUNS_DIR  = "/kaggle/working/runs"     # registry + results + adapters (saved as output)

In [ ]:
# ---- bootstrap: fetch the code at GIT_REF and install it ---------------------
import os, subprocess, sys

os.environ["PYTHONUNBUFFERED"] = "1"

def sh(cmd, check=True):
    # Stream a shell command's output into this cell; raise if it fails.
    print("$", cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="", flush=True)
    rc = p.wait()
    if check and rc:
        raise RuntimeError(f"command failed with exit code {rc}: {cmd}")
    return rc

if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    sh(f"git clone --quiet {REPO_URL} {REPO_DIR}")
sh(f"git -C {REPO_DIR} fetch --quiet origin {GIT_REF}")
sh(f"git -C {REPO_DIR} checkout --quiet --force FETCH_HEAD")
sh(f"git -C {REPO_DIR} log -1 --oneline")

sh(f"pip install -q -e {REPO_DIR}")
sh('pip install -q "peft>=0.12" "trl>=0.12" "bitsandbytes>=0.45"')
sh("pip freeze > /kaggle/working/requirements.lock")   # exact versions, saved as output

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

In [ ]:
# ---- environment + secrets ---------------------------------------------------
from ror import kaggle

print(kaggle.gpu_summary())
found = kaggle.load_secrets(["HF_TOKEN"])          # returns names only, never values
print("secrets available:", found or "none")
if "HF_TOKEN" not in found:
    print("note: HF_TOKEN is not attached; gated models (Llama-3.1) will fail to download")

In [ ]:
# ---- data: the attached ror-data dataset -------------------------------------
import json
from pathlib import Path

DATA_DIR = kaggle.ensure_data_dir()
if DATA_DIR is None:
    print("ror-data dataset not attached; building it from the pinned upstream sources instead")
    sh("python -u scripts/prepare_data.py --data-dir /tmp/ror-data --no-zip")
    DATA_DIR = Path("/tmp/ror-data")
os.environ["ROR_DATA_DIR"] = str(DATA_DIR)          # inherited by every script below

manifest = json.loads((DATA_DIR / "ror_data_manifest.json").read_text())
print(f"data: {DATA_DIR}  (preprocess v{manifest['preprocess_version']}, "
      f"built by code {manifest.get('code_commit')})")
for ds, info in manifest["datasets"].items():
    print(f"  {ds:10s}", ", ".join(f"{s}={v['n']}" for s, v in info["splits"].items()))
print("  clean set:", "present" if (DATA_DIR / "clean_set" / "clean.jsonl").exists()
      else "not built yet (clean-split experiments are skipped until it exists)")

In [ ]:
# ---- restore the registry from earlier versions of this notebook -------------
report = kaggle.restore_runs(Path(RUNS_DIR))
print("restored from:", report["sources"] or "nothing attached (fresh start)")
print(f"files copied: {report['files_copied']}, result rows added: {report['results_added']}")
if report["interrupted"]:
    print("interrupted runs (resume from checkpoint if attempts remain):", report["interrupted"])
sh(f"python -u scripts/status.py --runs-dir {RUNS_DIR}")

In [ ]:
# ---- tests -------------------------------------------------------------------
if RUN_TESTS:
    sh("python -m pytest -q -p no:cacheprovider")

In [ ]:
# ---- plan (always a dry run) -------------------------------------------------
FILTERS = " ".join(f"--{flag} {value}" for flag, value in
                   [("only", ONLY_ARM), ("model", MODEL), ("split", SPLIT), ("seed", SEED)]
                   if value is not None)
sh(f"python -u scripts/run_suite.py {SUITE} --runs-dir {RUNS_DIR} {FILTERS} --dry-run")

In [ ]:
# ---- run ---------------------------------------------------------------------
if DRY_RUN:
    print("DRY_RUN = True: nothing executed. Set DRY_RUN = False in CONFIG to run the plan above.")
else:
    sh(f"python -u scripts/run_suite.py {SUITE} --runs-dir {RUNS_DIR} {FILTERS}", check=False)

In [ ]:
# ---- results -----------------------------------------------------------------
from IPython.display import Markdown, display

sh(f"python -u scripts/status.py --runs-dir {RUNS_DIR}")
sh(f"python -u scripts/aggregate_results.py --runs-dir {RUNS_DIR}")
table = Path(RUNS_DIR) / "ablation_table.md"
if table.exists():
    display(Markdown(table.read_text()))

## Saving and resuming

- A committed version's output is everything in `/kaggle/working`: `runs/` (registry, `results.jsonl`, per-run logs, adapters, checkpoints) and `requirements.lock`.
- **Next session:** attach that output as an input (setup step 4) before running. Completed experiments are skipped. A run that was killed mid-way is marked *failed* and retried, resuming training from its latest checkpoint; each retry uses one of its `max_attempts` (2 by default).
- A run that stops because code is still a stub (`NotImplementedError` raised inside `ror/`) does **not** use up an attempt; it is simply picked up again once the code exists.
- Commit `requirements.lock` to the repo once the environment is stable (CLAUDE.md section 7).